# Guided Lab：自訂短期記憶狀態與使用工具讀寫

**時間：20 分鐘｜情境：電商客服的訂單授權檢查**

登入系統已驗證客戶身分，應用程式取得 `user_id="user_123"`。

你要讓客服 Agent 在同一對話緒中保存這個身分，先檢查訂單授權，再決定是否查詢訂單狀態。

完成後，你能：

1. 擴充 `AgentState`，並在建立及首次呼叫 Agent 時傳入自訂狀態(欄位)。
2. 在工具中用 `ToolRuntime` 讀取登入者編號(`user_id`)，再回傳 `Command` 更新短期記憶內的檢查紀錄相關欄位。
3. 用工具請求(tool call request)、工具結果與狀態快照(snapshot)，解釋第二輪沒有傳入 `user_id` 時如何檢查授權。

先備能力：已理解 [memory_1.md](memory_1.md)講義內容，能閱讀 Python 字典、類別與函式；已理解 `create_agent`、`@tool`、`InMemorySaver` 與 `thread_id`。

| 時間 | 活動 |
|---|---|
| 0–2 分鐘 | 閱讀情境與元件責任 |
| 2–10 分鐘 | 完成七個填空、寫第一輪預測並執行 |
| 10–15 分鐘 | 追蹤工具訊息與自訂狀態 |
| 15–18 分鐘 | 改查他人的訂單，先預測再比較 |
| 18–20 分鐘 | 解釋資料流並繳交 |




## 1. 辨識元件與資料來源

### Agent 的短期記憶(Agent 狀態)

自訂的短期記憶狀態
`OrderAgentState` 繼承 `AgentState`，保留 `messages`，並加入以下兩個欄位：

- `user_id`：登入系統提供的使用者識別碼。應用程式在第一輪寫入。
- `last_auth_check`：工具寫入的最後一次檢查紀錄。`LastAuthCheck` 是保存 `order_id` 與 `timestamp` 的資料類別。

### 工具函數如何取得自訂狀態及更新狀態

工具函數中的參數
`ToolRuntime[None, OrderAgentState]` 表示本例沒有自訂執行環境資料，工具取得的 state 的 Memory 結構為 `OrderAgentState`。
- LangChain 執行工具時自動注入 `runtime`；模型只需提供訂單編號。

工具函數會回傳
`Command(update=...)` 以更新短期記憶。
工具函數不能直接修改短期記憶內容。

工具回傳 `Command` 時，也要在 `update["messages"]` 放入 `ToolMessage`，把授權結果加入訊息歷史。
- `ToolMessage.tool_call_id` 需要和原工具請求的 `id` 配對。

### 程式中的元件與責任如下

| 元件 | 責任與資料來源 |
|---|---|
| 應用程式 | 將可信任的登入結果寫入第一輪 state |
| checkpointer | 按 `thread_id` 保存與恢復訊息及自訂欄位 |
| `check_order_authorization` 工具| 從 `runtime.state` 讀取 `user_id`，查模擬授權表，更新檢查紀錄並回傳授權結果 |
| `get_order_status` 工具 | 按 `order_id` 查模擬訂單表 |
| LLM 模型 | 依提示及工具結果決定下一次工具呼叫或回覆 |

注意:
- 自訂欄位預設不會自動加入模型的提示上下文。



In [2]:
# 載入 API Key
from dotenv import load_dotenv
from pathlib import Path
import os

# dotenv_path = Path("E:/.env")
# load_dotenv(dotenv_path)

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("找不到 OPENAI_API_KEY")

### 設定與訂單工具

下格匯入本次需要的元件，設定 chat 模型，並建立 `get_order_status`工具。

工具 `get_order_status` 輸入為 `order_id`，輸出為訂單狀態字典。

列表變數 `tool_events` 是本機觀察清單，記錄工具實際執行的順序與參數，以提供執行結過的觀查。

修改變數 `model_name` 設定成自己想用的模型名稱。

In [ ]:

import json
from dataclasses import dataclass
from datetime import datetime, timezone
from langchain.agents import AgentState, create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

# 可修改成你自己的模型名稱
model_name = "openai:gpt-5-nano"

authenticated_user_id = "user_123"  # 模擬登入系統已驗證的結果
order_authorizations = {
    "user_123": ["A1024", "B2048"],
    "user_456": ["C4096"],
}
tool_events = []

@tool
def get_order_status(order_id: str) -> dict:
    """依訂單編號查詢模擬訂單狀態與送達日期。"""
    tool_events.append({"tool": "get_order_status", "order_id": order_id})
    orders = {
        "A1024": {"status": "Shipped", "estimated_delivery": "2023-07-25"},
        "B2048": {"status": "Processing", "estimated_delivery": "2023-07-30"},
        "C4096": {"status": "Delivered", "estimated_delivery": "2023-07-20"},
    }
    return orders.get(
        order_id, {"status": "Order ID not found", "estimated_delivery": None}
    )

print("設定與模擬訂單工具已備妥。")


## 2. 完成自訂狀態與授權工具

以下工具接收 `order_id`，從 runtime 讀取登入者，檢查模擬授權表，再輸出 `Command`以更新短期記憶。

填入 **F01–F04**。`__Fxx__` 是填空標記，須以 Python 程式取代；未填完會失敗，請先完成再執行。

| 填空 | 提示 |
|---|---|
| F01 | 繼承哪個既有 state，才能保留 `messages`？ |
| F02 | 從工具 runtime 的 state 取得登入者欄位 |
| F03 | 用哪個物件將 `update` 交回 Agent？ |
| F04 | 從 runtime 取得目前工具請求的識別碼 |


In [ ]:
from typing import TypedDict


class LastAuthCheck(TypedDict):
    order_id: str
    timestamp: str

class OrderAgentState(__F01__):  # F01
    user_id: str
    last_auth_check: LastAuthCheck

@tool
def check_order_authorization(
    order_id: str, runtime: ToolRuntime[None, OrderAgentState]
) -> Command:
    """檢查目前登入者是否可查此訂單，並更新最後一次檢查紀錄。"""
    user_id = __F02__  # F02：從 state 讀取，不由模型傳入
    authorized_order_ids = order_authorizations.get(user_id, [])
    is_authorized = order_id in authorized_order_ids
    tool_events.append({
        "tool": "check_order_authorization",
        "order_id": order_id,
        "user_id": user_id,
        "authorized": is_authorized,
        "tool_call_id": runtime.tool_call_id,
    })
    return __F03__(  # F03
        update={
            "last_auth_check": LastAuthCheck(
                order_id=order_id,
                timestamp=datetime.now(timezone.utc).isoformat(),
            ),
            "messages": [
                ToolMessage(
                    content=json.dumps({
                        "order_id": order_id,
                        "authorized": is_authorized,
                    }),
                    tool_call_id=__F04__,  # F04
                )
            ],
        }
    )


### 建立 Agent，指定狀態結構與檢查點儲存器

下格提供系統提示與記憶儲存器。填入 **F05–F06**，建立同時具備訂單工具、自訂 state 與短期記憶的 Agent。

系統提示要求先授權再查詢，用來觀察模型的工具呼叫順序。正式訂單服務仍須在服務端強制檢查授權。

重做整個實驗時，重新啟動 kernel，再從第一格依序執行。這會建立全新的 checkpointer，清除舊對話及觀察紀錄。


In [ ]:
authorized_system_prompt = """
你是電商客服助理，請以繁體中文簡潔回答。
每次收到訂單查詢，都必須先呼叫 check_order_authorization，
即使歷史中已有授權結果，也要針對本次訂單重新檢查。
只有它回傳 authorized=true，才可呼叫 get_order_status 查詢同一訂單。
若 authorized=false，直接告知無法存取，不呼叫 get_order_status。
不要猜測訂單狀態；依工具結果回覆，測試資料的日期照原樣回答。
"""

# 建立 checkpointer
checkpointer = InMemorySaver()

# 建立 Agent
agent = create_agent(
    model=model_name,
    tools=[check_order_authorization, get_order_status],
    system_prompt=authorized_system_prompt,
    state_schema=__F05__,  # F05：指定自訂狀態類別
    checkpointer=__F06__,  # F06：傳入已建立的記憶儲存器
)

thread_config = {"configurable": {"thread_id": "custom-state-order-lab"}}
print("Agent 已建立。先完成下一格預測，再執行第一輪。")


## 3. 第一輪：初始化登入者並查自己的訂單

第一輪輸入包含訊息與 `user_id`。填入 **F07**：使用應用程式已取得的登入結果初始化 state。

**先編輯下格的預測，再執行 `invoke()`。**


### 第一輪預測（執行前填寫）

1. `check_order_authorization` 將從哪個欄位取得登入者？該值最初由誰提供？
   - 我的預測：＿＿＿＿
2. `user_123` 能否查詢 `A1024`？預測兩個工具的執行順序。
   - 我的預測：＿＿＿＿
3. 呼叫完成後，`last_auth_check` 與 `messages` 各會新增什麼資訊？
   - 我的預測：＿＿＿＿


In [ ]:
initial_state = {
    "messages": [{"role": "user", "content": "請查詢訂單 A1024 的狀態。"}],
    "user_id": __F07__,  # F07：來自登入系統的結果
}
start_1 = len(tool_events)
response_1 = agent.invoke(initial_state, config=thread_config)
events_1 = tool_events[start_1:]
print(response_1["messages"][-1].content)


## 4. 追蹤第一輪證據

觀察程式 `show_evidence` 會顯示實際工具紀錄、回傳 state、工具呼叫請求及工具結果。

底下程式中:
- `tool_calls` 的 `args` 是模型提出的工具參數；`id` 是該請求的識別碼。
- `ToolMessage` 是工具結果；其 `tool_call_id` 可與請求 `id` 配對。
- `agent.get_state(thread_config).values` 是該對話緒最新檢查點的 state，由教師提供的程式讀取。

本機工具紀錄顯示實際讀到的登入者。訊息與快照可核對請求、結果及保存的欄位；它們不呈現模型的內部推理。


In [ ]:
def show_evidence(response, events, message_start=0):
    print("實際工具執行紀錄：")
    for event in events:
        print(event)
    print("回傳 user_id：", response.get("user_id"))
    print("回傳 last_auth_check：", response.get("last_auth_check"))
    print("本輪工具請求與工具結果：")
    for message in response["messages"][message_start:]:
        for call in getattr(message, "tool_calls", []):
            print("請求：", call)
        if isinstance(message, ToolMessage):
            print("結果：", message.content, "| tool_call_id：", message.tool_call_id)


In [ ]:
# 顯示第一輪的工具呼叫與結果
show_evidence(response_1, events_1)

# 取得最新的快照並顯示 user_id 與 last_auth_check
snapshot_1 = agent.get_state(thread_config).values
print("最新檢查點 user_id：", snapshot_1.get("user_id"))
print("最新檢查點 last_auth_check：", snapshot_1.get("last_auth_check"))

# 顯示給模型的工具參數結構。runtime 是由執行框架注入的。
print("授權工具的模型參數結構：", check_order_authorization.tool_call_schema.model_json_schema())


### 第一輪核對（執行後填寫）

| 項目 | 實際觀察或具體證據 |
|---|---|
| 授權工具實際讀到的 `user_id` | ＿＿＿＿ |
| 授權工具請求的 `args` 有哪些欄位？ | ＿＿＿＿ |
| 授權結果的 `tool_call_id` 對應哪一個請求 `id`？ | ＿＿＿＿ |
| `last_auth_check` 的 `order_id` 與 `timestamp` | ＿＿＿＿ |
| 最新檢查點保存的登入者與檢查紀錄 | ＿＿＿＿ |

用一句話說明：模型沒有在工具請求中提供 `user_id`，工具仍能取得它的原因。

我的解釋：＿＿＿＿


## 5. 改變一個條件：改查他人的訂單

將訂單從 `A1024` 改成 `C4096`，但登入者、Agent、授權表與 `thread_id` 保持相同。這是同一對話緒的第二輪，訊息歷史會繼續累積。

本輪只傳入新訊息，省略 `user_id`。執行前，先根據第一輪的檢查點與授權表回答下列問題。


### 第二輪預測（執行前填寫）

1. 本輪輸入省略 `user_id`。工具預計讀到什麼值？資料從哪裡恢復？
   - 我的預測：＿＿＿＿
2. `C4096` 的授權結果與後續工具呼叫會如何改變？
   - 我的預測：＿＿＿＿
3. 檢查完成後，`last_auth_check` 預計記錄哪張訂單？先前的工具訊息還會存在嗎？
   - 我的預測：＿＿＿＿


In [ ]:
# 第一輪已保存的訊息數，用來只顯示第二輪新增的工具訊息。
message_start_2 = len(snapshot_1["messages"])
start_2 = len(tool_events)
response_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "請查詢訂單 C4096 的狀態。"}]},
    config=thread_config,
)
# 第二輪的工具訊息(只顯示新增的部分，在最後一個位置)
events_2 = tool_events[start_2:]
print(response_2["messages"][-1].content)

# 顯示第二輪的工具呼叫與結果
show_evidence(response_2, events_2, message_start=message_start_2)

# 取得最新的快照並顯示 user_id 與 last_auth_check
snapshot_2 = agent.get_state(thread_config).values
print("最新檢查點 user_id：", snapshot_2.get("user_id"))
print("最新檢查點 last_auth_check：", snapshot_2.get("last_auth_check"))


### 比較與解釋（執行後填寫）

| 項目 | 第一輪 A1024 | 第二輪 C4096 |
|---|---|---|
| 工具實際讀到的登入者 | ＿＿＿＿ | ＿＿＿＿ |
| 授權結果 | ＿＿＿＿ | ＿＿＿＿ |
| 實際呼叫的工具及順序 | ＿＿＿＿ | ＿＿＿＿ |
| 最新檢查點的 `last_auth_check.order_id` | ＿＿＿＿ | ＿＿＿＿ |

第二輪省略登入者，授權工具仍取得同一個值。請使用 **`thread_id`、checkpointer、`runtime.state`、授權表、`Command.update`、`ToolMessage`**，說明：

1. 登入者資料如何送到授權工具？
2. 拒絕存取是根據哪一份資料？
3. 哪一條資料路徑保存檢查紀錄？哪一條路徑把授權結果交給模型？

我的解釋（3–4 句）：＿＿＿＿

請引用本輪的一項實際證據：＿＿＿＿


## 6. 繳交與參考資料

繳交完成的學生 notebook，包含七個填空、兩輪預測、執行輸出、觀察表與最後解釋。若程式失敗，保留錯誤輸出並註記停在哪一步。

教師分別檢查「能完成建構與呼叫程序」及「能解釋資料來源與更新路徑」。模型措辭可能不同，請依本輪工具紀錄與 state 核對；若呼叫順序不符提示，也要記錄實際結果。

- [本章講義：memory_2.md](memory_2.md)
- [前章講義：memory_1.md](memory_1.md)
- [LangChain：自訂短期記憶](https://docs.langchain.com/oss/python/langchain/short-term-memory#customizing-agent-memory)
- [LangChain：工具讀寫 state 與 ToolRuntime](https://docs.langchain.com/oss/python/langchain/tools#access-context)
- [LangGraph：JsonPlusSerializer](https://reference.langchain.com/python/langgraph.checkpoint/serde/jsonplus/JsonPlusSerializer)

本檔交付時未保存執行輸出；教師版提供的是預期觀察。20 分鐘為設計估計，須由課堂試教確認。
